# Classical Axis 2 Report: Comparing Traditional Classifiers on PubMedQA


## 1. Purpose of This Notebook

This notebook focuses on **Classical Axis 2**, which means we keep the text representation mostly the same and compare different traditional machine learning classifiers.

The task is a medical question answering classification problem using the **PubMedQA** dataset. Each example contains:

- a medical question,
- one or more context sentences from biomedical abstracts,
- a long answer,
- a final label: `yes`, `no`, or `maybe`.

In this part, the goal is not to generate a free-text answer. Instead, the model predicts one of the three answer labels.

The main research question is: If we use classical machine learning methods, which classifier or training strategy works best for predicting PubMedQA answer labels?


## 2. Overall Code Pipeline

The notebook is designed to run independently. It extracts the necessary setup from the full group notebook and keeps only the parts needed for the Classical Axis 2 comparison.

The workflow is:

1. Load required Python libraries.
2. Load the PubMedQA dataset.
3. Clean and prepare the text.
4. Split the data into training/CV and test sets.
5. Build TF-IDF text features.
6. Train a baseline model.
7. Compare five Classical Axis 2 models.
8. Summarise and visualise the results.


## 3. Dataset Preparation

The code loads `ori_pqal.json`, which contains 1,000 labelled PubMedQA examples.

Each example is converted into a row in a Pandas dataframe with columns such as:

- `pmid`: the PubMed ID,
- `question`: the medical question,
- `context`: the biomedical abstract/context,
- `long_answer`: the provided long answer,
- `label`: the final decision label.

The labels are mapped into numbers:

| Label | Numeric ID |
|---|---:|
| `no` | 0 |
| `maybe` | 1 |
| `yes` | 2 |

This numeric format is needed because scikit-learn classifiers expect numerical target labels.


## 4. Text Cleaning and Input Representation

The text is cleaned by:

- converting everything to lowercase,
- removing unusual symbols,
- normalising extra spaces.

For this Axis 2 experiment, the main input text is:

```text
question + context
```

In the code, this is stored as `q_ctx`.

This is important because Axis 2 is mainly about comparing **classifiers and training strategies**, not comparing different text inputs. By keeping the input representation fixed, the comparison is fairer.


## 5. Train/Test and Cross-Validation Setup

The dataset is split into:

- **500 examples** for cross-validation,
- **500 examples** for final testing.

The 500 cross-validation examples are further split into **10 folds**.

This means the model is trained and validated 10 times. Each time, one fold is used for validation and the other nine folds are used for training.

This gives a more reliable estimate than using only one train/validation split.

The final test set is kept separate and is used only for final evaluation.


## 6. Evaluation Metrics

The notebook reports two main metrics:

### Accuracy

Accuracy measures the percentage of total predictions that are correct.

For example, if a model gets 250 out of 500 test examples correct, its accuracy is:

```text
250 / 500 = 0.50
```

### Macro-F1

Macro-F1 is more useful for this task because the labels are imbalanced.

In PubMedQA, the `yes` class is much more common than `maybe`. A model can get reasonable accuracy by mostly predicting `yes`, but that does not mean it understands all classes well.

Macro-F1 calculates F1 for each class separately and then averages them equally.

This means `no`, `maybe`, and `yes` are treated as equally important.

For this coursework, **Macro-F1 is the more important metric**.


## 7. Baseline Model in the Axis1

The baseline model uses:

- TF-IDF word features,
- unigrams and bigrams,
- Logistic Regression,
- balanced class weights.

The baseline gives:

| Model | CV Macro-F1 | Test Accuracy | Test Macro-F1 |
|---|---:|---:|---:|
| Baseline TF-IDF + Logistic Regression | 0.3432 ± 0.0525 | 0.5060 | 0.3482 |

This gives a reference point. Any Classical Axis 2 model should ideally improve over this baseline.

The baseline performs reasonably on the `yes` class but struggles badly with the `maybe` class.


## 8. Classical Axis 2 Models Compared

Five classical models or training strategies are tested.

### A2a: Tuned Logistic Regression with Word + Character TF-IDF

This model improves the feature representation by combining:

- word-level TF-IDF,
- character-level TF-IDF.

Character features can help capture small word patterns, spelling variations, and biomedical terms.

The Logistic Regression `C` value is tuned using grid search.

### A2b: Calibrated Linear SVM with Maybe Threshold Tuning

Linear SVM is often strong for text classification.

However, SVM scores are not probabilities by default. The code therefore calibrates the SVM so it can output class probabilities.

Then it tunes a threshold for the `maybe` class. The idea is to encourage the model to predict `maybe` when it is uncertain.

### A2c: Multinomial Naive Bayes

Naive Bayes is a simple and fast classical text classifier.

It assumes features are conditionally independent, which is not fully true for language, but it can still work well as a lightweight baseline.

The smoothing parameter `alpha` is tuned.

### A2d: SMOTE + Logistic Regression

SMOTE is an oversampling method.

It creates synthetic training examples for minority classes, such as `maybe`, so the classifier sees a more balanced training set.

The aim is to reduce the model's bias toward the majority class.

### A2e: Soft Voting Ensemble

This combines three models:

- Logistic Regression,
- calibrated SVM,
- Naive Bayes.

Instead of using only one classifier, the ensemble averages their predicted probabilities and chooses the most likely class.

The hope is that different models make different mistakes, so combining them may improve robustness.


## 9. Main Results

The Classical Axis 2 results are:

| Model | CV Macro-F1 | Test Accuracy | Test Macro-F1 |
|---|---:|---:|---:|
| A2a: Logistic Regression word+char | 0.3248 ± 0.0424 | 0.5320 | 0.3377 |
| A2b: Calibrated SVM word+char | 0.2773 ± 0.0398 | 0.5440 | 0.2642 |
| A2c: Multinomial Naive Bayes | 0.3108 ± 0.0273 | 0.5160 | 0.3224 |
| A2d: SMOTE + Logistic Regression | 0.3650 ± 0.0470 | 0.5180 | 0.3464 |
| A2e: Soft Voting Ensemble | N/A | 0.5320 | 0.3243 |

The best Classical Axis 2 model selected by cross-validation is:

```text
A2d: SMOTE + Logistic Regression
```

It has the highest CV Macro-F1:

```text
0.3650 ± 0.0470
```

However, on the held-out test set, its Macro-F1 is:

```text
0.3464
```

This is very close to the baseline test Macro-F1 of:

```text
0.3482
```

So although SMOTE helps during cross-validation, it does not clearly beat the baseline on the final test set.


## 10. Per-Class Interpretation

The most important observation is that all classical models struggle with the `maybe` class.

From the final results:

| Model | F1 no | F1 maybe | F1 yes |
|---|---:|---:|---:|
| Baseline | 0.3583 | 0.0571 | 0.6292 |
| A2a: LR word+char | 0.3139 | 0.0345 | 0.6647 |
| A2b: Cal-SVM | 0.0933 | 0.0000 | 0.6995 |
| A2c: MNB | 0.2794 | 0.0351 | 0.6528 |
| A2d: SMOTE + LR | 0.3660 | 0.0308 | 0.6423 |
| A2e: Ensemble | 0.2656 | 0.0357 | 0.6715 |

The `yes` class is easiest for the models. This is probably because:

- it has the most training examples,
- many biomedical abstracts provide positive findings,
- the models learn common words associated with positive answers.

The `maybe` class is hardest. This is probably because:

- it has fewer examples,
- it represents uncertainty,
- uncertain biomedical evidence is difficult to detect with simple TF-IDF features,
- the language of `maybe` can overlap with both `yes` and `no`.


## 11. Why Accuracy Can Be Misleading

A2b has the highest test accuracy:

```text
0.5440
```

But it has a poor Macro-F1:

```text
0.2642
```

This happens because A2b mostly predicts the majority `yes` class. It gets many `yes` examples correct, which increases accuracy, but it fails on the minority classes, especially `maybe`.

This is why Macro-F1 is more meaningful than accuracy for this task.


## 12. What the Results Mean

The Classical Axis 2 experiment shows that changing the classifier alone does not solve the PubMedQA task.

The models can learn some useful patterns from TF-IDF features, especially for `yes` and sometimes `no`, but they do not understand biomedical uncertainty very well.

SMOTE gives the best cross-validation result, which suggests that class imbalance is part of the problem. However, its test result is still close to the baseline, so oversampling alone is not enough.

The soft voting ensemble also does not improve Macro-F1. This suggests that the individual classical models are making similar mistakes, especially on the `maybe` class.


## 13. Final Conclusion

Among the Classical Axis 2 variants, **A2d: SMOTE + Logistic Regression** achieves the highest cross-validation Macro-F1, so it is selected as the best classical Axis 2 model according to the validation protocol.

However, its held-out test Macro-F1 is very close to the simple reference baseline:

```text
Reference baseline Test Macro-F1: 0.3482
A2d Test Macro-F1:                0.3464
```


Because the baseline uses word-level TF-IDF while A2d uses a stronger word + character TF-IDF setup with SMOTE, this should not be interpreted as a perfectly controlled classifier-only comparison. Instead, it shows that the more advanced classical setup does not lead to a clear improvement on the held-out test set.

Therefore, the main conclusion is: Classical TF-IDF-based models provide useful reference systems, but in this experiment they show limited ability to handle PubMedQA reliably, especially the minority maybe class.

For stronger performance, richer biomedical language representations, such as transformer-based models, are likely needed because they can capture semantic meaning and uncertainty better than sparse TF-IDF features.